In [10]:
import os
import re
import csv
from datetime import datetime, timedelta

import pandas as pd
import requests
import soundfile as sf
import numpy as np

# =====================================================
# CONFIG
# =====================================================

TOKEN = "srtfbicBifmyImxvLSyktBaWlfmE1EL1"

MANIFEST = "download_manifest.csv"

OUTPUT_ROOT = "/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s"

SEGMENT_SECONDS = 2

ETHOGRAM_START = datetime(2026, 4, 23, 0, 0, 0)
ETHOGRAM_END = datetime(2026, 6, 21, 23, 59, 59)

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

# =====================================================
# CHECK OUTPUT DIRECTORY
# =====================================================

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)

if not os.access(OUTPUT_ROOT, os.W_OK):

    raise RuntimeError(
        f"\nOUTPUT DIRECTORY NOT WRITABLE:\n{OUTPUT_ROOT}\n"
    )

# =====================================================
# METADATA STORAGE
# =====================================================

metadata_rows = []

# =====================================================
# EXTRACT TIMESTAMP
# =====================================================

def extract_timestamp(filename):

    patterns = [

        # 2026-04-24_09-40-00
        r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",

        # 20260424_094000
        r"(\d{8}_\d{6})"

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            filename
        )

        if not m:
            continue

        timestamp_text = m.group(1)

        try:

            return datetime.strptime(
                timestamp_text,
                "%Y-%m-%d_%H-%M-%S"
            )

        except:
            pass

        try:

            return datetime.strptime(
                timestamp_text,
                "%Y%m%d_%H%M%S"
            )

        except:
            pass

    return None

# =====================================================
# ETHOGRAM FILTER
# =====================================================

def in_ethogram_period(filename):

    ts = extract_timestamp(
        filename
    )

    if ts is None:

        return False

    return (
        ETHOGRAM_START
        <= ts
        <= ETHOGRAM_END
    )

# =====================================================
# DOWNLOAD FILE
# =====================================================

def download_file(
    file_id,
    out_file
):

    url = (
        f"https://api.box.com/2.0/files/"
        f"{file_id}/content"
    )

    r = requests.get(
        url,
        headers=headers,
        stream=True,
        allow_redirects=True
    )

    r.raise_for_status()

    with open(
        out_file,
        "wb"
    ) as f:

        for chunk in r.iter_content(
            1024 * 1024
        ):

            if chunk:

                f.write(chunk)

# =====================================================
# SEGMENT WAV
# =====================================================

def segment_wav(
    wav_file,
    original_filename
):

    start_time = extract_timestamp(
        original_filename
    )

    if start_time is None:

        print(
            f"NO TIMESTAMP FOUND: {original_filename}"
        )

        return

    audio, sr = sf.read(
        wav_file
    )

    if len(
        audio.shape
    ) > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    samples_per_segment = (
        sr * SEGMENT_SECONDS
    )

    n_segments = (
        len(audio)
        //
        samples_per_segment
    )

    for i in range(
        n_segments
    ):

        start_sample = (
            i *
            samples_per_segment
        )

        end_sample = (
            start_sample +
            samples_per_segment
        )

        segment = audio[
            start_sample:end_sample
        ]

        segment_time = (
            start_time
            +
            timedelta(
                seconds=
                i *
                SEGMENT_SECONDS
            )
        )

        segment_name = (
            segment_time.strftime(
                "%Y-%m-%d_%H-%M-%S"
            )
            +
            ".wav"
        )

        out_file = os.path.join(
            OUTPUT_ROOT,
            segment_name
        )

        try:

            sf.write(
                out_file,
                segment,
                sr
            )

        except Exception as e:

            print(
                f"WRITE FAILED: {out_file}"
            )

            print(e)

            continue

        metadata_rows.append([

            segment_name,

            original_filename,

            segment_time.isoformat()

        ])

# =====================================================
# LOAD MANIFEST
# =====================================================

manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "file_id": str
    }
)

print()
print(
    f"Files in manifest: "
    f"{len(manifest)}"
)

# =====================================================
# PROCESS FILES
# =====================================================

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    filename = row["file_name"]

    # ------------------------------------------
    # DATE FILTER
    # ------------------------------------------

    if not in_ethogram_period(
        filename
    ):

        print(
            "SKIPPED:",
            filename
        )

        continue

    temp_file = (
        "__temp.wav"
    )

    try:

        print()

        print(
            "Downloading:",
            filename
        )

        download_file(
            file_id,
            temp_file
        )

        segment_wav(
            temp_file,
            filename
        )

    except Exception as e:

        print(
            "FAILED:",
            filename
        )

        print(e)

    finally:

        if os.path.exists(
            temp_file
        ):

            try:
                os.remove(
                    temp_file
                )
            except:
                pass

# =====================================================
# SAVE INDEX
# =====================================================

index_file = os.path.join(
    OUTPUT_ROOT,
    "segment_index.csv"
)

with open(
    index_file,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([

        "segment_file",

        "parent_file",

        "timestamp"

    ])

    writer.writerows(
        metadata_rows
    )

# =====================================================
# SUMMARY
# =====================================================

print()
print("=" * 40)
print("DONE")
print("=" * 40)

print(
    f"Segments created: "
    f"{len(metadata_rows)}"
)

print(
    f"Index saved to:\n"
    f"{index_file}"
)


Files in manifest: 1873
SKIPPED: Data_20260218_140952.wav
SKIPPED: Data_20260218_141152.wav
SKIPPED: Data_20260218_141352.wav
SKIPPED: Data_20260218_141552.wav
SKIPPED: Data_20260218_141752.wav
SKIPPED: Data_20260218_141952.wav
SKIPPED: Data_20260218_142152.wav
SKIPPED: Data_20260218_142352.wav
SKIPPED: Data_20260218_142552.wav
SKIPPED: Data_20260218_142752.wav
SKIPPED: Data_20260218_142952.wav
SKIPPED: Data_20260218_143152.wav
SKIPPED: Data_20260218_143352.wav
SKIPPED: Data_20260218_143552.wav
SKIPPED: Data_20260218_143752.wav
SKIPPED: Data_20260218_143952.wav
SKIPPED: Data_20260218_144152.wav
SKIPPED: Data_20260218_144244.wav
SKIPPED: Data_20260218_144444.wav
SKIPPED: Data_20260218_144644.wav
SKIPPED: Data_20260218_144844.wav
SKIPPED: Data_20260218_145044.wav
SKIPPED: Data_20260218_145244.wav
SKIPPED: Data_20260218_145444.wav
SKIPPED: Data_20260218_145644.wav
SKIPPED: Data_20260218_145844.wav
SKIPPED: Data_20260218_150044.wav
SKIPPED: Data_20260218_150244.wav
SKIPPED: Data_20260218_

In [11]:
import os
import re
import csv
from datetime import datetime, timedelta

import pandas as pd
import requests
import soundfile as sf
import numpy as np

# ==========================================================
# BOX CONFIG
# ==========================================================

CLIENT_ID = ""
CLIENT_SECRET = ""
REFRESH_TOKEN = ""

TOKEN = None

# ==========================================================
# PATHS
# ==========================================================

MANIFEST = "download_manifest.csv"

OUTPUT_ROOT = "/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s"

CHECKPOINT_FILE = os.path.join(
    OUTPUT_ROOT,
    "processed_files.csv"
)

INDEX_FILE = os.path.join(
    OUTPUT_ROOT,
    "segment_index.csv"
)

SEGMENT_SECONDS = 2

# ==========================================================
# ETHOGRAM PERIOD
# ==========================================================

ETHOGRAM_START = datetime(
    2026, 4, 23, 0, 0, 0
)

ETHOGRAM_END = datetime(
    2026, 6, 21, 23, 59, 59
)

# ==========================================================
# CREATE OUTPUT DIRECTORY
# ==========================================================

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)

if not os.access(
    OUTPUT_ROOT,
    os.W_OK
):
    raise RuntimeError(
        f"\nOUTPUT DIRECTORY NOT WRITABLE:\n{OUTPUT_ROOT}\n"
    )

# ==========================================================
# TOKEN REFRESH
# ==========================================================

def refresh_access_token():

    global TOKEN
    global REFRESH_TOKEN

    url = "https://api.box.com/oauth2/token"

    payload = {
        "grant_type": "refresh_token",
        "refresh_token": REFRESH_TOKEN,
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET
    }

    r = requests.post(
        url,
        data=payload,
        timeout=60
    )

    r.raise_for_status()

    data = r.json()

    TOKEN = data["access_token"]

    if "refresh_token" in data:
        REFRESH_TOKEN = data["refresh_token"]

    print("\nToken refreshed successfully\n")

# ==========================================================
# HEADERS
# ==========================================================

def get_headers():

    return {
        "Authorization": f"Bearer {TOKEN}"
    }

# ==========================================================
# TIMESTAMP EXTRACTION
# ==========================================================

def extract_timestamp(filename):

    patterns = [

        r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",

        r"(\d{8}_\d{6})"

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            filename
        )

        if not m:
            continue

        value = m.group(1)

        try:

            return datetime.strptime(
                value,
                "%Y-%m-%d_%H-%M-%S"
            )

        except:
            pass

        try:

            return datetime.strptime(
                value,
                "%Y%m%d_%H%M%S"
            )

        except:
            pass

    return None

# ==========================================================
# ETHOGRAM FILTER
# ==========================================================

def in_ethogram_period(filename):

    ts = extract_timestamp(
        filename
    )

    if ts is None:
        return False

    return (
        ETHOGRAM_START
        <= ts
        <= ETHOGRAM_END
    )

# ==========================================================
# CHECKPOINT FILE
# ==========================================================

if not os.path.exists(
    CHECKPOINT_FILE
):

    with open(
        CHECKPOINT_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "file_id",
            "file_name"
        ])

# ==========================================================
# LOAD CHECKPOINT
# ==========================================================

processed = set()

with open(
    CHECKPOINT_FILE,
    "r",
    encoding="utf-8"
) as f:

    next(f, None)

    for line in f:

        line = line.strip()

        if not line:
            continue

        parts = line.split(",")

        if len(parts) >= 1:

            processed.add(
                parts[0]
            )

print(
    f"Already completed: "
    f"{len(processed)} files"
)

# ==========================================================
# SEGMENT INDEX
# ==========================================================

if not os.path.exists(
    INDEX_FILE
):

    with open(
        INDEX_FILE,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "segment_file",
            "parent_file",
            "timestamp"
        ])

# ==========================================================
# DOWNLOAD WITH AUTO RETRY
# ==========================================================

def download_file(
    file_id,
    out_file,
    retries=5
):

    global TOKEN

    url = (
        f"https://api.box.com/2.0/files/"
        f"{file_id}/content"
    )

    for attempt in range(
        retries
    ):

        try:

            r = requests.get(
                url,
                headers=get_headers(),
                stream=True,
                allow_redirects=True,
                timeout=300
            )

            if r.status_code == 401:

                print(
                    "\n401 Unauthorized"
                )

                print(
                    "Refreshing token..."
                )

                refresh_access_token()

                continue

            r.raise_for_status()

            with open(
                out_file,
                "wb"
            ) as f:

                for chunk in r.iter_content(
                    chunk_size=1024 * 1024
                ):

                    if chunk:
                        f.write(chunk)

            return

        except Exception as e:

            print(
                f"Download retry "
                f"{attempt + 1}/{retries}"
            )

            print(e)

    raise RuntimeError(
        f"Failed download: {file_id}"
    )

# ==========================================================
# SEGMENTATION
# ==========================================================

def segment_wav(
    wav_file,
    original_filename
):

    start_time = extract_timestamp(
        original_filename
    )

    if start_time is None:

        print(
            "NO TIMESTAMP:",
            original_filename
        )

        return 0

    audio, sr = sf.read(
        wav_file
    )

    if len(audio.shape) > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    samples_per_segment = (
        sr *
        SEGMENT_SECONDS
    )

    n_segments = (
        len(audio)
        //
        samples_per_segment
    )

    created = 0

    with open(
        INDEX_FILE,
        "a",
        newline="",
        encoding="utf-8"
    ) as idx_file:

        writer = csv.writer(
            idx_file
        )

        for i in range(
            n_segments
        ):

            start_sample = (
                i *
                samples_per_segment
            )

            end_sample = (
                start_sample +
                samples_per_segment
            )

            segment = audio[
                start_sample:end_sample
            ]

            ts = (
                start_time
                +
                timedelta(
                    seconds=i *
                    SEGMENT_SECONDS
                )
            )

            segment_name = (
                ts.strftime(
                    "%Y-%m-%d_%H-%M-%S"
                )
                + ".wav"
            )

            out_file = os.path.join(
                OUTPUT_ROOT,
                segment_name
            )

            sf.write(
                out_file,
                segment,
                sr
            )

            writer.writerow([
                segment_name,
                original_filename,
                ts.isoformat()
            ])

            created += 1

    return created

# ==========================================================
# INITIAL TOKEN
# ==========================================================

refresh_access_token()

# ==========================================================
# LOAD MANIFEST
# ==========================================================

manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "file_id": str
    }
)

print(
    f"\nManifest contains "
    f"{len(manifest)} files\n"
)

# ==========================================================
# MAIN PROCESSING LOOP
# ==========================================================

total_segments = 0

for _, row in manifest.iterrows():

    file_id = str(
        row["file_id"]
    )

    filename = (
        row["file_name"]
    )

    if file_id in processed:

        continue

    if not in_ethogram_period(
        filename
    ):

        continue

    temp_file = "__temp.wav"

    try:

        print(
            "Downloading:",
            filename
        )

        download_file(
            file_id,
            temp_file
        )

        created = segment_wav(
            temp_file,
            filename
        )

        total_segments += created

        with open(
            CHECKPOINT_FILE,
            "a",
            newline="",
            encoding="utf-8"
        ) as f:

            writer = csv.writer(
                f
            )

            writer.writerow([
                file_id,
                filename
            ])

            f.flush()
            os.fsync(
                f.fileno()
            )

        processed.add(
            file_id
        )

        print(
            f"Completed: "
            f"{filename} "
            f"({created} segments)"
        )

    except Exception as e:

        print(
            "FAILED:",
            filename
        )

        print(e)

    finally:

        if os.path.exists(
            temp_file
        ):

            try:

                os.remove(
                    temp_file
                )

            except:
                pass

# ==========================================================
# FINISH
# ==========================================================

print("\n")
print("=" * 60)
print("DONE")
print("=" * 60)

print(
    f"Processed files: "
    f"{len(processed)}"
)

print(
    f"Segments created: "
    f"{total_segments}"
)

Already completed: 0 files


HTTPError: 400 Client Error: Bad Request for url: https://api.box.com/oauth2/token

In [15]:
import os
import re
import csv
from datetime import datetime, timedelta

import pandas as pd
import requests
import soundfile as sf
import numpy as np

# =====================================================
# CONFIG
# =====================================================

TOKEN = "Y9vrlsr9ESBYbQNPK1931TwCWFooguAf"

MANIFEST = "download_manifest.csv"

OUTPUT_ROOT = "/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s"

SEGMENT_SECONDS = 2

ETHOGRAM_START = datetime(2026, 4, 23, 0, 0, 0)
ETHOGRAM_END = datetime(2026, 6, 21, 23, 59, 59)

# Resume exactly where the previous run failed
RESUME_FROM = " LUW6548_20260521_145000.wav"

headers = {
    "Authorization": f"Bearer {TOKEN}"
}

# =====================================================
# CHECK OUTPUT DIRECTORY
# =====================================================

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True
)

if not os.access(OUTPUT_ROOT, os.W_OK):

    raise RuntimeError(
        f"\nOUTPUT DIRECTORY NOT WRITABLE:\n{OUTPUT_ROOT}\n"
    )

# =====================================================
# METADATA STORAGE
# =====================================================

metadata_rows = []

# =====================================================
# EXTRACT TIMESTAMP
# =====================================================

def extract_timestamp(filename):

    patterns = [

        r"(\d{4}-\d{2}-\d{2}_\d{2}-\d{2}-\d{2})",

        r"(\d{8}_\d{6})"

    ]

    for pattern in patterns:

        m = re.search(
            pattern,
            filename
        )

        if not m:
            continue

        timestamp_text = m.group(1)

        try:

            return datetime.strptime(
                timestamp_text,
                "%Y-%m-%d_%H-%M-%S"
            )

        except:
            pass

        try:

            return datetime.strptime(
                timestamp_text,
                "%Y%m%d_%H%M%S"
            )

        except:
            pass

    return None

# =====================================================
# ETHOGRAM FILTER
# =====================================================

def in_ethogram_period(filename):

    ts = extract_timestamp(
        filename
    )

    if ts is None:

        return False

    return (
        ETHOGRAM_START
        <= ts
        <= ETHOGRAM_END
    )

# =====================================================
# DOWNLOAD FILE
# =====================================================

def download_file(
    file_id,
    out_file
):

    url = (
        f"https://api.box.com/2.0/files/"
        f"{file_id}/content"
    )

    r = requests.get(
        url,
        headers=headers,
        stream=True,
        allow_redirects=True
    )

    r.raise_for_status()

    with open(
        out_file,
        "wb"
    ) as f:

        for chunk in r.iter_content(
            1024 * 1024
        ):

            if chunk:

                f.write(chunk)

# =====================================================
# SEGMENT WAV
# =====================================================

def segment_wav(
    wav_file,
    original_filename
):

    start_time = extract_timestamp(
        original_filename
    )

    if start_time is None:

        print(
            f"NO TIMESTAMP FOUND: {original_filename}"
        )

        return

    audio, sr = sf.read(
        wav_file
    )

    if len(audio.shape) > 1:

        audio = np.mean(
            audio,
            axis=1
        )

    samples_per_segment = (
        sr * SEGMENT_SECONDS
    )

    n_segments = (
        len(audio)
        //
        samples_per_segment
    )

    for i in range(
        n_segments
    ):

        start_sample = (
            i *
            samples_per_segment
        )

        end_sample = (
            start_sample +
            samples_per_segment
        )

        segment = audio[
            start_sample:end_sample
        ]

        segment_time = (
            start_time
            +
            timedelta(
                seconds=i *
                SEGMENT_SECONDS
            )
        )

        segment_name = (
            segment_time.strftime(
                "%Y-%m-%d_%H-%M-%S"
            )
            +
            ".wav"
        )

        out_file = os.path.join(
            OUTPUT_ROOT,
            segment_name
        )

        try:

            sf.write(
                out_file,
                segment,
                sr
            )

        except Exception as e:

            print(
                f"WRITE FAILED: {out_file}"
            )

            print(e)

            continue

        metadata_rows.append([
            segment_name,
            original_filename,
            segment_time.isoformat()
        ])

# =====================================================
# LOAD MANIFEST
# =====================================================

manifest = pd.read_csv(
    MANIFEST,
    dtype={
        "file_id": str
    }
)

print()
print(
    f"Files in manifest: "
    f"{len(manifest)}"
)

# =====================================================
# RESUME LOGIC
# =====================================================

resume = False

# =====================================================
# PROCESS FILES
# =====================================================

for _, row in manifest.iterrows():

    file_id = row["file_id"]

    filename = row["file_name"]

    # ------------------------------------------
    # RESUME FROM FAILED FILE
    # ------------------------------------------

    if not resume:

        if filename == RESUME_FROM:

            resume = True

            print()
            print(
                f"RESUMING FROM: {filename}"
            )
            print()

        else:

            continue

    # ------------------------------------------
    # ETHOGRAM FILTER
    # ------------------------------------------

    if not in_ethogram_period(
        filename
    ):

        continue

    temp_file = "__temp.wav"

    try:

        print(
            "Downloading:",
            filename
        )

        download_file(
            file_id,
            temp_file
        )

        segment_wav(
            temp_file,
            filename
        )

    except Exception as e:

        print(
            "FAILED:",
            filename
        )

        print(e)

        break

    finally:

        if os.path.exists(
            temp_file
        ):

            try:

                os.remove(
                    temp_file
                )

            except:
                pass

# =====================================================
# SAVE INDEX
# =====================================================

index_file = os.path.join(
    OUTPUT_ROOT,
    "segment_index.csv"
)

with open(
    index_file,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([

        "segment_file",

        "parent_file",

        "timestamp"

    ])

    writer.writerows(
        metadata_rows
    )

# =====================================================
# SUMMARY
# =====================================================

print()
print("=" * 50)
print("DONE")
print("=" * 50)

print(
    f"Segments created: "
    f"{len(metadata_rows)}"
)

print(
    f"Index saved to:\n"
    f"{index_file}"
)


Files in manifest: 1873

DONE
Segments created: 0
Index saved to:
/media/feliciano/Aux/AI_AFS_DATASET/segmented_2s/segment_index.csv
